# MAgent2 BenchMARL Baselines

Train and evaluate established BenchMARL baselines on MAgent2 `battle_v4` with 64 red agents versus 64 blue agents. MAgent2 is wrapped through BenchMARL/TorchRL; VMAS vectorization is available for VMAS tasks, not this MAgent2 task.


In [1]:
from pathlib import Path
import sys
import torch
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "discrete_action_space").is_dir() and (path / "requirements.txt").exists():
            return path
    raise RuntimeError("Could not find the SRE-DQN repo root from the current notebook directory.")

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from discrete_action_space.mean_field_dsrq.benchmarl_magent2 import DEFAULT_TASK_CONFIG, DEFAULT_USE_MASK
from discrete_action_space.mean_field_dsrq.notebook_utils import (
    baseline_rollout_video_from_notebook,
    run_benchmarl_algorithm
)

TASK_CONFIG = {**DEFAULT_TASK_CONFIG, "env_name": "battle_v4", "map_size": 40, "max_cycles": 400}
BENCHMARL_USE_MASK = DEFAULT_USE_MASK
TOTAL_FRAMES = 20_000
FRAMES_PER_BATCH = 1_000
MAX_STEPS = 40
N_ENVS_PER_WORKER = 1
SEED = 42
SAVE_FOLDER = ROOT / "discrete_action_space" / "mean_field_dsrq" / "runs" / "benchmarl_magent2_notebooks"
SAVE_FOLDER.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

common_kwargs = dict(
    task_config=TASK_CONFIG,
    use_mask=BENCHMARL_USE_MASK,
    seed=SEED,
    total_frames=TOTAL_FRAMES,
    frames_per_batch=FRAMES_PER_BATCH,
    n_envs_per_worker=N_ENVS_PER_WORKER,
    save_folder=SAVE_FOLDER,
    sampling_device="cpu",
    train_device=DEVICE,
    buffer_device="cpu",
    parallel_collection=True,
)
TASK_CONFIG

{'env_name': 'battle_v4',
 'map_size': 40,
 'max_cycles': 400,
 'minimap_mode': False,
 'extra_features': False}

## MAPPO

In [2]:
mappo_result = run_benchmarl_algorithm("mappo", **common_kwargs)
mappo_result

/home/wowthecoder/SRE-DQN/venv/lib/python3.12/site-packages/torchrl/envs/libs/pettingzoo.py:281: UserWarning: PettingZoo in TorchRL is tested using version == 1.24.3 , If you are using a different version and are experiencing compatibility issues,please raise an issue in the TorchRL github.
  warnings.warn(
/home/wowthecoder/SRE-DQN/venv/lib/python3.12/site-packages/torchrl/collectors/_base.py:1045: DeprecationWarning: SyncDataCollector has been deprecated and will be removed in v0.13. Please use Collector instead.
  warnings.warn(
  0%|                                                                                            | 0/20 [00:00<?, ?it/s]

2026-05-29 11:00:59,713 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([1000]) shape [END]
2026-05-29 11:01:01,702 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([1000]) shape [END]


mean return = -16.582536697387695:   5%|██▍                                             | 1/20 [02:00<38:15, 120.83s/it]



Experiment was closed gracefully




KeyboardInterrupt: 

## MAPPO Evaluation Rollout


In [ ]:
baseline_rollout_video_from_notebook(
    mappo_result,
    max_steps=50,
    fps=8,
    deterministic=True,
    title="MAPPO evaluation rollout",
)


## IPPO

In [ ]:
ippo_result = run_benchmarl_algorithm("ippo", **common_kwargs)
ippo_result

## IPPO Evaluation Rollout


In [ ]:
baseline_rollout_video_from_notebook(
    ippo_result,
    max_steps=50,
    fps=8,
    deterministic=True,
    title="IPPO evaluation rollout",
)


## IQL

In [ ]:
iql_result = run_benchmarl_algorithm("iql", **common_kwargs)
iql_result

## IQL Evaluation Rollout


In [ ]:
baseline_rollout_video_from_notebook(
    iql_result,
    max_steps=50,
    fps=8,
    deterministic=True,
    title="IQL evaluation rollout",
)


## Comparison

In [ ]:
baseline_results = [mappo_result, ippo_result, iql_result]
pd.DataFrame(baseline_results)